# RAG Pipeline

Build a complete Retrieval-Augmented Generation pipeline: load local documents, chunk, embed, store in a vector database, and query with context grounding.

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import numpy as np

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

print("Models ready!")

Models ready!


## Embedding
At the core of every LLM is embedding, a process that converts a string into a vector representation. 
This allows us to compare the similarity between two sentences based on their vector distance. 
The goal is to compare sentences based on their meaning rather than just their words.

In [2]:
embed_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vec_query = embed_model.embed_query("Hello World")

In [3]:
type(vec_query)

list

In [13]:
len(vec_query) # different embedding has different length

3072

In [14]:
contents = [
    "Cuacanya sangat dingin hari ini",
    "Buku terbaru yang ada di toko sangat bagus",
    "Taman itu berada di daerah Jakarta Kota"
]
vec_contents = np.array([embed_model.embed_query(_) for _ in contents])

In [15]:
vec_contents.shape

(3, 3072)

In [29]:
query = "Di dekat sini terdapat area hijau yang rimbun untuk bermain"
# query = "Taman itu berada di daerah Jakarta Kota"
vec_query = np.array(embed_model.embed_query(query))

In [30]:
np.linalg.norm(vec_query-vec_contents, axis=1) # smaller is better

array([0.85206355, 0.84464507, 0.78592992])

## Vector Storage (Qdrant In-Memory)

Vector databases store and search over high-dimensional vector embeddings using distance metrics (cosine, Euclidean, etc.). They enable semantic search — finding content by *meaning* rather than keyword matching. Popular options include Qdrant, Pinecone, Weaviate, and Chroma. Here we use Qdrant in-memory mode, no external server needed.

In [16]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="demo_kuncie",
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="demo_kuncie",
    embedding=embed_model,
)
# print(f"Vector store created with {len(all_chunks)} vectors")

## Storing Embedding

In [17]:
sentences = [
    "Nasi goreng adalah makanan favorit di Indonesia",
    "Sate ayam dengan bumbu kacang sangat lezat",
    "Rendang daging sapi yang dimasak lama sangat empuk",
    "Es teh manis segar diminum saat siang hari",
    "Jus alpukat campur susu coklat adalah minuman favorit",
]
vector_store.add_texts(sentences)
print(f"Total vectors in database: {client.count('demo_kuncie').count}")

Total vectors in database: 5


## Retrieval

In [19]:
query = "Minuman apa yang cocok untuk cuaca panas?"
results = vector_store.similarity_search_with_score(query, k=4)
for doc, score in results:
    print(f"Score: {score:.4f} | {doc.page_content}")

Score: 0.7508 | Es teh manis segar diminum saat siang hari
Score: 0.6851 | Jus alpukat campur susu coklat adalah minuman favorit
Score: 0.6211 | Sate ayam dengan bumbu kacang sangat lezat
Score: 0.5981 | Rendang daging sapi yang dimasak lama sangat empuk


## Limited Knowledge of External Data

A bare LLM (without RAG) only knows what it learned during training — it has no access to internal company documents like **Peraturan Perusahaan**, HR policies, or proprietary API docs. Even if the LLM was trained on public Indonesian labor laws, it cannot answer questions about *this specific company's* policies (e.g., "Berapa hari cuti di PT Lore Ipsum?"). RAG bridges this gap by injecting relevant private documents into the prompt.

In [26]:
query = "Berapa hari cuti tahunan karyawan PT Lore Ipsum Tbk?. Jika tidak mengetahui, jawab saja tidak tahu."
print(llm.invoke(query).text)

Saya tidak mengetahui secara pasti berapa hari cuti tahunan karyawan PT Lore Ipsum Tbk, karena "Lore Ipsum" biasanya merupakan teks contoh (placeholder) dan bukan nama perusahaan yang nyata.

Jika perusahaan tersebut memang ada, kebijakan cuti tahunan biasanya diatur dalam **Perjanjian Kerja (PK)**, **Peraturan Perusahaan (PP)**, atau **Perjanjian Kerja Bersama (PKB)** masing-masing perusahaan, dengan tetap mengacu pada ketentuan minimal **UU Cipta Kerja (UU No. 6 Tahun 2023)** yang menetapkan hak cuti tahunan minimal **12 hari kerja** setelah karyawan bekerja selama 12 bulan secara terus-menerus.


In [28]:
from pathlib import Path

doc_path = Path("../docs/dummy-knowledge/Peraturan-Perusahaan-LoreIpsum.txt")
content = doc_path.read_text(encoding="utf-8")
content = "\n".join(line for line in content.splitlines() if line.strip())

print(f"Loaded: {doc_path.name} ({len(content)} chars)")
print("\n--- First 500 characters ---")
print(content[:500])

Loaded: Peraturan-Perusahaan-LoreIpsum.txt (62207 chars)

--- First 500 characters ---
﻿
PERATURAN PERUSAHAAN
PT LORE IPSUM TBK
DAFTAR ISI
Halaman
Daftar Isi	2
Mukadimah	4
Bab	I	KETENTUAN UMUM
Pasal 1. Pengertian dan Istilah	5
Pasal 2. Maksud dan Tujuan	8
Pasal 3. Ruang Lingkup Peraturan Perusahaan	8
Bab	II	HUBUNGAN KERJA
Pasal 4. Formasi	9
Pasal 5. Persyaratan Dalam Penerimaan Karyawan	9
Pasal 6. Status Hubungan Kerja dan Penggolongan Karyawan	10
Pasal 7. Masa Percobaan	11
Pasal 8. Pengangkatan	11
Pasal 9. Penetapan Jabatan	12
Pasal 10. Perubahan Jabatan	12
Pasal 11. Jenis Peruba


## Document Chunking

Large documents don't fit in the LLM's context window efficiently. We split them into smaller overlapping chunks.

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Strip excessive blank lines
content = "\n".join(line for line in content.splitlines() if line.strip())

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=250,
    separators=[""],
)

chunks = splitter.split_text(content)
print(f"Total chunks: {len(chunks)}")

Total chunks: 248


In [36]:
print(chunks[50])

ihak lain.
        e) Tidak terdaftar sebagai anggota atau ikut dalam kepesertaan /keanggotaan organisasi terlarang.
        f) Bersedia mentaati peraturan/tata tertib yang berlaku dan yang akan diberlakukan oleh perusahaan.
        g) Tidak memiliki hubungan keluarga sedarah dan atau ikatan perkawinan dengan karyawan lainnya (dalam satu unit usaha).
        h) Bebas pengaruh alkohol, narkoba dan obat-obatan terlarang.
    4. Penerimaan karyawan dilakukan melalui prosedur rekrutmen yang ditetapk


## Store Chunks to Vector Database

Drop the previous food/drink demo data, then embed and store the Peraturan Perusahaan chunks.

In [37]:
# Drop old data and recreate collection
client.delete_collection("demo_kuncie")
client.create_collection(
    collection_name="demo_kuncie",
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
)
vector_store = QdrantVectorStore(
    client=client,
    collection_name="demo_kuncie",
    embedding=embed_model,
)
print("Collection reset — ready for new data")

Collection reset — ready for new data


In [41]:
# Insert chunks one by one with progress
import time
total = len(chunks)
batch = []
for i, chunk in enumerate(chunks):
    batch.append(chunk)
    if (i + 1) % 10 == 0 or i == total - 1:
        vector_store.add_texts(batch)
        batch = []
        print(f"  Embedded {i+1}/{total} chunks...")
        print("Sleeping")
        time.sleep(6)
        print("Wake up")

if batch:
    vector_store.add_texts(batch)
    batch = []
print(f"Done. Total vectors: {client.count('demo_kuncie').count}")

  Embedded 10/248 chunks...
Sleeping
Wake up
  Embedded 20/248 chunks...
Sleeping
Wake up
  Embedded 30/248 chunks...
Sleeping
Wake up
  Embedded 40/248 chunks...
Sleeping
Wake up
  Embedded 50/248 chunks...
Sleeping
Wake up
  Embedded 60/248 chunks...
Sleeping
Wake up
  Embedded 70/248 chunks...
Sleeping
Wake up
  Embedded 80/248 chunks...
Sleeping
Wake up
  Embedded 90/248 chunks...
Sleeping
Wake up
  Embedded 100/248 chunks...
Sleeping
Wake up
  Embedded 110/248 chunks...
Sleeping
Wake up
  Embedded 120/248 chunks...
Sleeping
Wake up
  Embedded 130/248 chunks...
Sleeping
Wake up
  Embedded 140/248 chunks...
Sleeping
Wake up
  Embedded 150/248 chunks...
Sleeping
Wake up
  Embedded 160/248 chunks...
Sleeping
Wake up
  Embedded 170/248 chunks...
Sleeping
Wake up
  Embedded 180/248 chunks...
Sleeping
Wake up
  Embedded 190/248 chunks...
Sleeping
Wake up
  Embedded 200/248 chunks...
Sleeping
Wake up
  Embedded 210/248 chunks...
Sleeping
Wake up
  Embedded 220/248 chunks...
Sleeping
Wake 

In [42]:
import pickle
from pathlib import Path

# Pull all vectors + payloads from Qdrant (no re-embedding)
all_points, _ = client.scroll(
    collection_name="demo_kuncie",
    with_vectors=True,
    with_payload=True,
    limit=10000,
)

data = [{"text": p.payload["page_content"], "embedding": p.vector} for p in all_points]

# Save to disk
output_path = Path("../data/embeddings_loreipsum.pkl")
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "wb") as f:
    pickle.dump(data, f)

print(f"Saved {len(data)} embeddings to {output_path}")

Saved 558 embeddings to ../data/embeddings_loreipsum.pkl


In [43]:
# with open("../data/embeddings_loreipsum.pkl", "rb") as f:
#     loaded_data = pickle.load(f)

# print(f"Loaded {len(loaded_data)} entries \u2014 no re-embedding needed")
# print(f"Example text: {loaded_data[0]['text'][:80]}...")
# print(f"Embedding dims: {len(loaded_data[0]['embedding'])}")

Loaded 558 entries — no re-embedding needed
Example text: ehari dan 40 (empat puluh) jam seminggu untuk 6 (enam) hari kerja dalam seminggu...
Embedding dims: 3072


## RAG Query

Retrieve relevant chunks from the vector database and use them as context for the LLM.

In [44]:
query = "Berapa hari cuti tahunan karyawan PT Lore Ipsum Tbk?"
results = vector_store.similarity_search_with_score(query, k=5)
print(f"=== Top 5 chunks for: '{query}' ===\n")
for i, (doc, score) in enumerate(results):
    print(f"--- Result {i+1} (score: {score:.4f}) ---")
    print(doc.page_content[:300] + "...\n")

=== Top 5 chunks for: 'Berapa hari cuti tahunan karyawan PT Lore Ipsum Tbk?' ===

--- Result 1 (score: 0.7491) ---
belas) bulan secara terus-menerus, berhak atas cuti tahunan selama 12 (dua belas) hari kerja dengan tetap mendapat upah. (gaji pokok ditambah tunjangan tetap).
        2. Perusahaan dapat menunda permohonan cuti tahunan karyawan, dan cuti tahunan tersebut dapat dibagi dalam beberapa bagian yang sala...

--- Result 2 (score: 0.7491) ---
belas) bulan secara terus-menerus, berhak atas cuti tahunan selama 12 (dua belas) hari kerja dengan tetap mendapat upah. (gaji pokok ditambah tunjangan tetap).
        2. Perusahaan dapat menunda permohonan cuti tahunan karyawan, dan cuti tahunan tersebut dapat dibagi dalam beberapa bagian yang sala...

--- Result 3 (score: 0.7490) ---
kan perusahaan.
        6. Berdasarkan Keputusan Direksi, cuti tahunan dapat diberikan secara massal.
    II. Cuti Besar
        1. Setiap karyawan yang telah bekerja selama 5 (lima) tahun secara terus – mener

In [47]:
from langchain_core.prompts import ChatPromptTemplate

context = "\n\n".join(doc.page_content for doc, _ in results)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Anda adalah asisten HR yang membantu menjawab pertanyaan berdasarkan Peraturan Perusahaan. Jawab hanya berdasarkan konteks yang diberikan. Jika jawaban tidak ada dalam konteks, katakan 'Saya tidak menemukan informasi tersebut dalam dokumen.'"),
    ("human", "Konteks:\n{context}\n\nPertanyaan: {query}".format(context=context, query=query)),
])

chain = prompt | llm
answer = chain.invoke({})
print(answer.text)

Setiap karyawan yang telah bekerja selama 12 (dua belas) bulan secara terus-menerus berhak atas cuti tahunan selama 12 (dua belas) hari kerja.
